# LoRA Fine-tuning: Llama-3.2-1B -- Banka Musteri Hizmetleri

Project 5'te sifirdan yazdigimiz LoRA kodunu **gercek bir LLM'e** uyguluyoruz.

**Senaryo:** Banka musterilerinin sorularini cevaplayan bir chatbot egitiyoruz.
Dataset: `bitext/Bitext-customer-support-llm-chatbot-training-dataset` -- 26.000 musteri-hizmetleri ornek

**Pipeline:**
1. Llama-3.2-1B yukle
2. Baseline inference -- banka sorusuna ilk cevap
3. `inject_lora()` -- sadece ~%1 parametre egitilecek
4. 500 musteri-hizmetleri ornegi ile fine-tune
5. Fine-tuned inference -- fark ne kadar?
6. `merge_lora_weights()` -- adapteri baz modele bak

> **Runtime:** Runtime > Change runtime type > **T4 GPU** sec

In [ ]:
# Kurulum
!pip install -q transformers datasets accelerate huggingface_hub
!git clone https://github.com/Mertgdk/ai-lora-finetuning.git

import sys
sys.path.insert(0, '/content/ai-lora-finetuning')
print('Kurulum tamamlandi.')

In [ ]:
import torch
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from huggingface_hub import login

from src.core.lora import inject_lora, count_parameters
from src.core.merge import merge_lora_weights

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# HuggingFace login -- kendi token'ini yaz
HF_TOKEN = 'hf_BURAYA_KENDI_TOKENINI_YAZ'
login(token=HF_TOKEN)
print('Login basarili.')

In [ ]:
# Model yukle
MODEL_ID = 'meta-llama/Llama-3.2-1B'
print(f'{MODEL_ID} yukleniyor... (~2.5GB, birkaç dakika sürebilir)')

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f'Model yuklendi. Toplam parametre: {total_params:,}')

In [ ]:
# Baseline inference -- fine-tuning ONCESI banka sorusuna cevap
TEMPLATE = (
    'You are a helpful bank customer support agent. '
    'Answer the customer question clearly and professionally.\n\n'
    '### Customer:\n{instruction}\n\n### Support Agent:\n'
)

def generate(model, tokenizer, prompt, max_new_tokens=150):
    model.eval()
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

TEST_QUESTIONS = [
    'My account has been blocked, what should I do?',
    'How can I apply for a credit card?',
    'I was charged twice for the same transaction.',
]

print('=== FINE-TUNING ONCESI ===\n')
baseline_answers = {}
for q in TEST_QUESTIONS:
    ans = generate(model, tokenizer, TEMPLATE.format(instruction=q))
    baseline_answers[q] = ans
    print(f'Musteri: {q}')
    print(f'Model:   {ans[:200]}')
    print('-' * 60)

In [ ]:
# LoRA inject -- q_proj ve v_proj hedef al
params_before = count_parameters(model)

inject_lora(model, target_modules=['q_proj', 'v_proj'], rank=8, alpha=16)

# GPU FIX: yeni LoRA katmanlarini GPU'ya tasima
# inject_lora CPU'da olusturuyor, device_map='auto' sonradan yuklenenleri otomatik tasimaz
for name, param in model.named_parameters():
    if 'lora' in name and param.device.type == 'cpu':
        param.data = param.data.to(device)

params_after = count_parameters(model)

print('=== PARAMETRE SAYILARI ===')
print(f'Toplam:          {params_after["total"]:>15,}')
print(f'Egitilecek:      {params_after["trainable"]:>15,}  ({params_after["trainable_pct"]:.4f}%)')
print(f'Dondurulmus:     {params_after["frozen"]:>15,}')
print(f'\nSadece %{params_after["trainable_pct"]:.2f} parametre egitiliyor -- geri kalan frozen!')

In [ ]:
# Musteri hizmetleri dataset -- 500 ornek
class CustomerSupportDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=256):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.samples = []
        template = (
            'You are a helpful bank customer support agent. '
            'Answer the customer question clearly and professionally.\n\n'
            '### Customer:\n{instruction}\n\n### Support Agent:\n{response}'
        )
        for item in data:
            instruction = item.get('instruction', '')
            response = item.get('response', '')
            if instruction.strip() and response.strip():
                self.samples.append(
                    template.format(instruction=instruction, response=response)
                )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.samples[idx],
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt',
        )
        return {
            'input_ids': enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
        }

print('Musteri hizmetleri dataset indiriliyor...')
raw = load_dataset(
    'bitext/Bitext-customer-support-llm-chatbot-training-dataset',
    split='train[:500]'
)
dataset = CustomerSupportDataset(raw, tokenizer, max_length=256)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

print(f'Dataset: {len(dataset)} ornek, {len(dataloader)} batch')
print(f'\nOrnek:')
print(f'  Musteri: {raw[0]["instruction"]}')
print(f'  Cevap:   {raw[0]["response"][:120]}...')

In [ ]:
# Egitim
EPOCHS = 3
LR = 2e-4

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR,
)

all_losses = []
epoch_losses = []

model.train()
print(f'Egitim basliyor: {EPOCHS} epoch | lr={LR} | batch=4')
print(f'Egitilen parametre: {params_after["trainable"]:,} ({params_after["trainable_pct"]:.4f}%)')
print('-' * 60)

for epoch in range(EPOCHS):
    epoch_loss = 0.0
    n_batches = 0

    for i, batch in enumerate(dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        labels = input_ids.clone()
        labels[labels == tokenizer.pad_token_id] = -100

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        all_losses.append(loss.item())
        n_batches += 1

        if (i + 1) % 20 == 0:
            print(f'  Epoch {epoch+1} | Batch {i+1}/{len(dataloader)} | Loss: {loss.item():.4f}')

    avg = epoch_loss / n_batches
    epoch_losses.append(avg)
    print(f'Epoch {epoch+1}/{EPOCHS} -- Ort. Loss: {avg:.4f}')
    print('-' * 60)

print(f'\nLoss: {epoch_losses[0]:.4f} -> {epoch_losses[-1]:.4f} ({100*(epoch_losses[0]-epoch_losses[-1])/epoch_losses[0]:.1f}% dusus)')

In [ ]:
# Loss grafigi
plt.figure(figsize=(10, 4))
plt.plot(all_losses, alpha=0.4, color='steelblue', label='Batch loss')
batches_per_epoch = len(dataloader)
colors = ['red', 'orange', 'green']
for i, el in enumerate(epoch_losses):
    plt.axhline(
        el,
        xmin=(i * batches_per_epoch) / len(all_losses),
        xmax=((i + 1) * batches_per_epoch) / len(all_losses),
        color=colors[i], linewidth=2.5,
        label=f'Epoch {i+1} avg: {el:.4f}'
    )
plt.xlabel('Batch')
plt.ylabel('Loss')
plt.title('LoRA Fine-tuning -- Banka Musteri Hizmetleri (Llama-3.2-1B)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ONCESI vs SONRASI karsilastirma
print('=' * 60)
print('FINE-TUNING ONCESI vs SONRASI')
print('=' * 60)

for q in TEST_QUESTIONS:
    prompt = TEMPLATE.format(instruction=q)
    new_ans = generate(model, tokenizer, prompt)
    print(f'\nMusteri: {q}')
    print(f'ONCESI:  {baseline_answers[q][:200]}')
    print(f'SONRASI: {new_ans[:200]}')
    print('-' * 60)

In [ ]:
# Merge -- LoRA adapter'i baz agirliga bak
test_input = tokenizer(TEMPLATE.format(instruction=TEST_QUESTIONS[0]), return_tensors='pt').to(device)
model.eval()

with torch.no_grad():
    logits_before = model(**test_input).logits.clone()

merge_lora_weights(model)

with torch.no_grad():
    logits_after = model(**test_input).logits

max_diff = (logits_before - logits_after).abs().max().item()
params_merged = count_parameters(model)

print('=== MERGE SONUCLARI ===')
print(f'Max logit farki:     {max_diff:.2e}')
print(f'Merge:               {"BASARILI" if max_diff < 1e-2 else "BASARISIZ"}')
print(f'Toplam param:        {params_merged["total"]:,}')
print(f'Trainable:           {params_merged["trainable_pct"]:.2f}% (LoRA gitti, tekrar full model)')

print('\n=== OZET ===')
print(f'Model:               Llama-3.2-1B ({total_params:,} parametre)')
print(f'Egitilen:            {params_after["trainable"]:,} ({params_after["trainable_pct"]:.4f}%)')
print(f'Dataset:             500 musteri-hizmetleri ornek')
print(f'Loss dususu:         {epoch_losses[0]:.4f} -> {epoch_losses[-1]:.4f}')
print(f'Merge dogrulama:     max diff = {max_diff:.2e}')